# CO-SPY pretrained — Eval nhanh FLUX.1-schnell (MSCOCO)

Chỉ eval **5.000 real MSCOCO val2017 + 5.000 fake FLUX.1-schnell** bằng **CO-SPY pretrained Fusion v1**.

- Không chạy toàn Co-SpyBench.
- Có progress bar.
- Xuất AP, Accuracy và CSV dự đoán.
- Bật **GPU** + **Internet** trên Kaggle.

In [1]:
from pathlib import Path
import os, sys, json, shutil, subprocess, time

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
REPO = WORK / "cospy"

BATCH_SIZE = 32
NUM_WORKERS = 4
N_REAL = 5000
N_FAKE = 5000
SEED = 1024

print("INPUT:", INPUT)

INPUT: /kaggle/input


In [2]:
# Clone source + dependencies
if not REPO.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/datnguyentrung/CO-SPY.git", str(REPO)],
        check=True,
    )

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "open-clip-torch==2.31.0",
    "diffusers==0.32.2",
    "transformers==4.46.3",
    "accelerate==1.0.1",
    "safetensors==0.5.3",
    "ftfy==6.2.3",
    "timm==1.0.15",
    "loguru==0.7.3",
], check=True)

print("Repo ready:", REPO)

Cloning into '/kaggle/working/cospy'...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
litellm 1.93.2 requires tokenizers<1.0,>=0.21.0, but you have tokenizers 0.20.3 which is incompatible.
urwid 5.0.1 requires wcwidth!=0.9.0,!=0.9.1,>=0.4, but you have wcwidth 0.2.14 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
prettytable 3.18.0 requires wcwidth>=0.3.5, but you have wcwidth 0.2.14 which is incompatible.


Repo ready: /kaggle/working/cospy


In [4]:
# CELL 3 — FAST: tìm weights + dataset mà không scan toàn /kaggle/input
from pathlib import Path
import os

INPUT = Path("/kaggle/input")
IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp"}

# =========================================================
# 1. PRETRAINED WEIGHTS
# =========================================================
PRETRAIN_ROOT = Path(
    "/kaggle/input/notebooks/datnguyentrung/dl-pretrained/sd-v1_4/sd-v1_4"
)

weights = {
    "artifact_weights.pth": PRETRAIN_ROOT / "artifact_weights.pth",
    "semantic_weights.pth": PRETRAIN_ROOT / "semantic_weights.pth",
    "fusion_weights.pth": PRETRAIN_ROOT / "fusion_weights.pth",
}

dst_dir = REPO / "pretrained" / "sd-v1_4"
dst_dir.mkdir(parents=True, exist_ok=True)

for name, src in weights.items():
    assert src.is_file(), f"Không tìm thấy {src}"

    dst = dst_dir / name

    if dst.exists() or dst.is_symlink():
        dst.unlink()

    dst.symlink_to(src)

    print("[WEIGHT]", name, "->", src)

# =========================================================
# 2. REAL — MSCOCO val2017 = 5000 ảnh
# =========================================================
REAL_DIR = Path(
    "/kaggle/input/datasets/datnguyentrung/co-spy-v1/"
    "mscoco2017/mscoco2017/val2017"
)

assert REAL_DIR.is_dir(), f"Không thấy: {REAL_DIR}"

real_images = sorted(
    p for p in REAL_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in IMG_EXTS
)

print("\n[REAL]")
print("Dir  :", REAL_DIR)
print("Count:", len(real_images))

# =========================================================
# 3. FAKE — chỉ tìm notebook output có chữ flux
# =========================================================
NOTEBOOK_ROOT = Path("/kaggle/input/notebooks/datnguyentrung")

flux_roots = [
    p for p in NOTEBOOK_ROOT.iterdir()
    if p.is_dir() and "flux" in p.name.lower()
]

print("\nFLUX notebook roots:")
for p in flux_roots:
    print(" -", p)

if not flux_roots:
    raise FileNotFoundError("Không tìm thấy notebook output FLUX.")

FLUX_ROOT = flux_roots[0]

# Chỉ scan folder FLUX (~5000 ảnh), rất nhanh
fake_images = sorted(
    p for p in FLUX_ROOT.rglob("*")
    if p.is_file() and p.suffix.lower() in IMG_EXTS
)

print("\n[FAKE]")
print("Root :", FLUX_ROOT)
print("Count:", len(fake_images))

# =========================================================
# 4. VALIDATE
# =========================================================
real_images = real_images[:5000]
fake_images = fake_images[:5000]

assert len(real_images) == 5000, f"Real: {len(real_images)}/5000"
assert len(fake_images) == 5000, f"Fake: {len(fake_images)}/5000"

print("\n✅ READY")
print("Real :", len(real_images))
print("Fake :", len(fake_images))
print("Total:", len(real_images) + len(fake_images))

[WEIGHT] artifact_weights.pth -> /kaggle/input/notebooks/datnguyentrung/dl-pretrained/sd-v1_4/sd-v1_4/artifact_weights.pth
[WEIGHT] semantic_weights.pth -> /kaggle/input/notebooks/datnguyentrung/dl-pretrained/sd-v1_4/sd-v1_4/semantic_weights.pth
[WEIGHT] fusion_weights.pth -> /kaggle/input/notebooks/datnguyentrung/dl-pretrained/sd-v1_4/sd-v1_4/fusion_weights.pth

[REAL]
Dir  : /kaggle/input/datasets/datnguyentrung/co-spy-v1/mscoco2017/mscoco2017/val2017
Count: 5000

FLUX notebook roots:
 - /kaggle/input/notebooks/datnguyentrung/dataset-flux-schnell-images

[FAKE]
Root : /kaggle/input/notebooks/datnguyentrung/dataset-flux-schnell-images
Count: 5000

✅ READY
Real : 5000
Fake : 5000
Total: 10000


In [5]:
# Load pretrained CO-SPY Fusion v1
import torch

os.chdir(REPO)
sys.path.insert(0, str(REPO))

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

from evaluate import Detector

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

t0 = time.time()
detector = Detector(
    device=device,
    mode="fusion",
    train_dataset="sd-v1_4",
    pretrain=True,
    ckpt=str(WORK / "eval_ckpt"),
    batch_size=BATCH_SIZE,
    fusion_head="v1",
)
model = detector.model
model.eval()

print(f"Model loaded in {(time.time()-t0)/60:.2f} min")

Device: cuda:0
GPU: Tesla T4


The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

I0000 00:00:1794470685.166813      50 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


open_clip_model.safetensors:   0%|          | 0.00/3.51G [00:00<?, ?B/s]

config.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

vae/diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

Model loaded in 1.27 min


In [ ]:
# Eval 10k images
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from sklearn.metrics import average_precision_score, accuracy_score

class EvalDataset(Dataset):
    def __init__(self, paths, labels, transform):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        with Image.open(self.paths[i]) as im:
            x = self.transform(im.convert("RGB"))
        return x, self.labels[i]

paths = real_images + fake_images
labels = [0]*len(real_images) + [1]*len(fake_images)

ds = EvalDataset(paths, labels, model.test_transform)
loader = DataLoader(
    ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2,
)

y_pred, y_true = [], []
start = time.time()

with torch.inference_mode():
    for images, yy in tqdm(loader, desc="CO-SPY Fusion eval", unit="batch"):
        pred = model.predict(images)
        y_pred.extend(float(v) for v in pred)
        y_true.extend(yy.tolist())

elapsed = time.time() - start
y_pred = np.asarray(y_pred, dtype=np.float64)
y_true = np.asarray(y_true, dtype=np.int64)

ap = average_precision_score(y_true, y_pred)
acc = accuracy_score(y_true, (y_pred >= 0.5).astype(np.int64))
real_ok = (y_pred[y_true == 0] < 0.5).mean()
fake_ok = (y_pred[y_true == 1] >= 0.5).mean()

result = {
    "model": "CO-SPY pretrained Fusion v1",
    "scope": "MSCOCO val2017 5k real vs FLUX.1-schnell 5k fake",
    "AP": float(ap),
    "AP_percent": float(ap*100),
    "Accuracy": float(acc),
    "Accuracy_percent": float(acc*100),
    "Real_correct_percent": float(real_ok*100),
    "Fake_detected_percent": float(fake_ok*100),
    "elapsed_minutes": float(elapsed/60),
    "images_per_second": float(len(y_true)/elapsed),
    "threshold": 0.5,
}

print("\n" + "="*64)
print(f"AP                 : {ap*100:.2f}%")
print(f"Accuracy           : {acc*100:.2f}%")
print(f"Real correctly real: {real_ok*100:.2f}%")
print(f"Fake detected AI   : {fake_ok*100:.2f}%")
print(f"Runtime            : {elapsed/60:.1f} min")
print(f"Throughput         : {len(y_true)/elapsed:.2f} img/s")
print("="*64)
print("Paper full FLUX.1-schnell reference: AP 95.52 | Acc 85.24")
print("Lưu ý: notebook này chỉ eval subset MSCOCO 5k+5k.")

CO-SPY Fusion eval:   0%|          | 0/313 [00:00<?, ?batch/s]

In [ ]:
# Save results
import csv

OUT = WORK / "cospy_flux_schnell_eval"
OUT.mkdir(parents=True, exist_ok=True)

with open(OUT/"result.json", "w", encoding="utf-8") as f:
    json.dump(result, f, indent=2, ensure_ascii=False)

with open(OUT/"predictions.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["path", "label", "score", "predicted_label"])
    for p, y, s in zip(paths, y_true.tolist(), y_pred.tolist()):
        w.writerow([str(p), y, s, int(s >= 0.5)])

print("Saved:")
print(OUT/"result.json")
print(OUT/"predictions.csv")